In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:


from pymongo import MongoClient
from bson import json_util
import json
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_cosmosdb_json, write_raw


# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("cosmosdb_ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

connection_string = dbutils.secrets.get(scope="retail-platform-dev", key=source_config["connection_string"])

# ============================================================
# 5. READ FROM COSMOSDB SERVER
# ============================================================
df = read_cosmosdb_json(
    spark,
    connection_string,
    source_config["database_name"],
    source_config["collection_name"],
    source_config["data_type"]
)

logger.info("Data successfully read from Cosmos DB")


# ============================================================
# 6. TARGET PATH
# ============================================================
base_path = target_config["base_path"]
folder = target_config["folder"]

target_path = f"{base_path}/{folder}/load_date={run_date}"
logger.info("writiing data to %s", target_path)
# ============================================================
# 7. WRITE RAW
# ============================================================

write_raw(df,target_path,target_config["file_format"],target_config["mode"])
logger.info("Data Succesfully written")
